In [ ]:
# 1) سحب الكود
import os, sys, subprocess

BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"
if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                     "https://github.com/jonsnow-org/Ttbik.git", CLONE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py"))
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)
print("جاهز في:", CODE_DIR)

In [ ]:
# 2) المكتبات الناقصة فقط
import subprocess
try:
    import datasets
except ImportError:
    subprocess.run(["pip", "install", "-q", "datasets"], check=True)
try:
    import tokenizers
except ImportError:
    subprocess.run(["pip", "install", "-q", "tokenizers"], check=True)
print("تم.")

In [ ]:
import os

print("جاري تفعيل الوضع غير المقيد...")

os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["DISABLE_CONTENT_FILTER"] = "1"
os.environ["UNRESTRICTED_MODE"] = "1"
os.environ["NO_MODERATION"] = "1"

def force_uncensored(x):
    return x

globals()["force_uncensored"] = force_uncensored
globals()["no_filter"] = force_uncensored
globals()["uncensored"] = force_uncensored

try:
    from dataset import TextSequenceDataset
    methods = ['filter_text', 'is_banned', 'clean_text', 'safety_filter', 'moderate', 'check_content', 'validate_text', 'ban_words']
    for method_name in methods:
        if hasattr(TextSequenceDataset, method_name):
            setattr(TextSequenceDataset, method_name, lambda self, *args, **kwargs: args[0] if args else True)
            print("تم تعطيل:", method_name)
    print("تم تطبيق التعطيل على TextSequenceDataset")
except Exception as e:
    print("لم يتم العثور على TextSequenceDataset:", e)

print("الوضع غير المقيد مفعّل بنجاح")


In [ ]:
# 3) جمع نص عربي حقيقي (ويكيبيديا، streaming)
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 20_000
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia", config_name="20231101.ar", text_field="text",
    output_dir="/kaggle/working/corpus/wikipedia_ar", max_documents=MAX_DOCUMENTS,
)
print(f"عدد ملفات الشحنات: {len(corpus_files)}")

In [ ]:
# 4) نفس أداة تقسيم النص المستخدمة سابقاً (لا تدرّب واحدة جديدة أبداً هنا)
from pathlib import Path
from text_tokenizer import ShamTextTokenizer, train_text_tokenizer
from model import TEXT_VOCAB_SIZE

found = (list(Path("/kaggle/input").rglob("sham_small_tokenizer.json"))
         + list(Path("/kaggle/input").rglob("nova_small_tokenizer.json")))
if found:
    tokenizer = ShamTextTokenizer.load(found[0])
    print(f"أعيد استخدام: {found[0]} (vocab={tokenizer.vocab_size})")
else:
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print(f"تدريب أداة جديدة (vocab={tokenizer.vocab_size}) — أول مرة فقط.")
tokenizer.save("/kaggle/working/sham_small_tokenizer.json")

In [ ]:
# 5) بناء بيانات التدريب
from dataset import TextSequenceDataset
SEQ_LEN = 1024
text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"عدد النوافذ: {len(text_dataset):,}")
assert len(text_dataset) >= 32

In [ ]:
# 6) استئناف من آخر نقطة حفظ
import torch
from pathlib import Path
from checkpoint import load_checkpoint
from train import build_optimizer

device = "cuda" if torch.cuda.is_available() else "cpu"
start_step, resume_optimizer = 0, None
ckpts = sorted(Path("/kaggle/input").rglob("step_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
last_ckpt = ckpts[-1]
model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
print(f"استؤنف من {last_ckpt} عند الخطوة {start_step:,} — {model.count_parameters():,} معامل")

In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_WARMUP_STEPS = 5
CALIBRATION_STEPS = 30

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 2 * 4), 2)
][: (CALIBRATION_WARMUP_STEPS + CALIBRATION_STEPS) * 4]
assert len(calib_batches) > CALIBRATION_WARMUP_STEPS, "لا توجد بيانات كافية للقياس -- كبّر MAX_DOCUMENTS في خلية جمع البيانات."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

def _calib_step(batch):
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()

for batch in calib_batches[:CALIBRATION_WARMUP_STEPS]:
    _calib_step(batch)

t0 = time.time()
steps_done = 0
for batch in calib_batches[CALIBRATION_WARMUP_STEPS:]:
    _calib_step(batch)
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed

MAX_TRAINING_HOURS = 8.5
realistic_steps_for_session = int(steps_per_second * MAX_TRAINING_HOURS * 3600 * 0.85)

print(f"سرعة حقيقية مقاسة الآن (بعد تجاوز {CALIBRATION_WARMUP_STEPS} خطوات إحماء): {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي يمكن إنجازه ضمن {MAX_TRAINING_HOURS} ساعة (بهامش أمان): {realistic_steps_for_session:,} خطوة")

In [ ]:
# 8) التدريب الحقيقي
import itertools
TOTAL_STEPS = max(realistic_steps, 200)
num_windows = len(text_dataset) - (len(text_dataset) % 4)

def _batch_iterator():
    while True:
        for b in range(0, num_windows, 4):
            yield torch.stack([text_dataset[i] for i in range(b, b+4)])

batches = itertools.islice(_batch_iterator(), TOTAL_STEPS)
train_cfg = TrainConfig(
    seq_len=SEQ_LEN, batch_size=4, grad_accum_steps=4, lr=3e-4,
    warmup_steps=max(50, TOTAL_STEPS // 100), total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints", checkpoint_every=200, log_every=20,
    max_wall_clock_seconds=MAX_TRAINING_HOURS * 3600,
)
from train import train
loss_history = train(model, batches, train_cfg, device=device, start_step=start_step, resume_optimizer=resume_optimizer)
print(f"انتهى على {len(loss_history):,} خطوة. أول خسارة: {sum(loss_history[:10])/10:.4f} — آخر خسارة: {sum(loss_history[-10:])/10:.4f}")

In [ ]:
# 9) حفظ نهائي
from checkpoint import save_checkpoint
final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"حُفظ عند الخطوة {final_step:,}")

In [ ]:
import json as _json
import os as _os
import subprocess as _subprocess
import shutil as _shutil
from pathlib import Path as _Path
from kaggle_secrets import UserSecretsClient as _UserSecretsClient

_subprocess.run(["pip", "install", "-q", "-U", "kaggle"], check=False)

KAGGLE_USERNAME = _UserSecretsClient().get_secret("KAGGLE_USERNAME")
KAGGLE_KEY = _UserSecretsClient().get_secret("KAGGLE_KEY")
DATASET_SLUG = f"{KAGGLE_USERNAME}/sham-checkpoint"

_os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
_os.environ["KAGGLE_KEY"] = KAGGLE_KEY

upload_dir = _Path("/kaggle/working/for_dataset_upload")
if upload_dir.exists():
    _shutil.rmtree(upload_dir)
(upload_dir / "checkpoints").mkdir(parents=True)
for ckpt in _Path("/kaggle/working/checkpoints").glob("*.pt"):
    _shutil.copy2(ckpt, upload_dir / "checkpoints" / ckpt.name)
_shutil.copy2("/kaggle/working/sham_small_tokenizer.json", upload_dir / "sham_small_tokenizer.json")

metadata = {"title": "sham-checkpoint", "id": DATASET_SLUG, "licenses": [{"name": "unknown"}]}
(upload_dir / "dataset-metadata.json").write_text(_json.dumps(metadata))

_list_result = _subprocess.run(["kaggle", "datasets", "list", "-m", "--csv"], capture_output=True, text=True)
_dataset_exists = DATASET_SLUG in (_list_result.stdout or "")

if _dataset_exists:
    result = _subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(upload_dir), "-m", f"auto-update at step {final_step:,}", "-r", "zip"],
        capture_output=True, text=True,
    )
else:
    result = _subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(upload_dir), "-r", "zip"],
        capture_output=True, text=True,
    )
_combined = (result.stdout or "") + (result.stderr or "")

if result.returncode == 0 and "error" not in _combined.lower():
    verb = "نُشرت نسخة جديدة إلى" if _dataset_exists else "تم إنشاء"
    print(f"{verb} {DATASET_SLUG} عند الخطوة {final_step:,} — الجلسة القادمة (المجدولة) ستلتقطها تلقائياً.")
elif "incompatible" in _combined.lower():
    print(
        "تنبيه: هذا الـDataset أُنشئ تحت وضع رفع غير متوافق. غيّري DATASET_SLUG أعلاه لاسم لم يُستخدم من قبل "
        "(مثلاً sham-checkpoint-v2) وأعيدي التشغيل. نقطة الحفظ آمنة في Output هذه الجلسة بغض النظر."
    )
    print(_combined)
else:
    print("تنبيه: فشل نشر النسخة الجديدة تلقائياً — نقطة الحفظ نفسها لا تزال محفوظة بأمان في Output هذه الجلسة. "
          "تحقق من صحة KAGGLE_USERNAME / KAGGLE_KEY.")
    print(_combined)